# 🌸 Iris Flower Classification Using Machine Learning

**Objective:** Classify an iris flower into **Setosa, Versicolor, or Virginica** using sepal and petal measurements.

This notebook covers EDA, visualisation, feature selection discussion, train/test split, three classifiers, evaluation, model comparison, and prediction.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load the Iris Dataset

The dataset is built into scikit-learn, so no download is required.

In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')

target_names = iris.target_names
df = X.copy()
df['species'] = y.map(dict(enumerate(target_names)))

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nNull values:')
print(df.isnull().sum())
print('\nDescriptive statistics:')
display(df.describe())
print('\nClass distribution:')
print(df['species'].value_counts())

## 4. Pairplot by Species

The pairplot shows the relationships between features and how well the species are separated.

In [ ]:
sns.pairplot(df, hue='species', diag_kind='hist')
plt.suptitle('Iris Feature Pairplot by Species', y=1.02)
plt.show()

## 5. Box Plots for Each Feature

In [ ]:
for feature in iris.feature_names:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x='species', y=feature, data=df)
    plt.title(f'{feature.title()} by Species')
    plt.xlabel('Species')
    plt.ylabel(feature.title())
    plt.show()

## 6. Feature Selection Discussion

From the visualisations, **petal length** and **petal width** are generally the most discriminative features because they show clearer separation between the three species. Sepal length and sepal width have more overlap. We will still train the models using all four features so that all available information can be used.

## 7. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

## 8. Train/Test Split

We use 80% for training and 20% for testing. Stratification keeps class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training samples:', X_train.shape[0])
print('Testing samples :', X_test.shape[0])

## 9. Model 1 — Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])
logistic_model.fit(X_train, y_train)
y_pred_logistic = logistic_model.predict(X_test)

## 10. Model 2 — K-Nearest Neighbours (KNN)

In [ ]:
knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

## 11. Model 3 — Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

## 12. Evaluate the Models

We use accuracy, confusion matrix, and classification report. The classification report includes precision, recall, and F1-score.

In [ ]:
models_predictions = {
    'Logistic Regression': y_pred_logistic,
    'KNN': y_pred_knn,
    'Random Forest': y_pred_rf
}

results = []
for name, predictions in models_predictions.items():
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, predictions)})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
display(results_df)

### Confusion Matrices

In [ ]:
for name, predictions in models_predictions.items():
    cm = confusion_matrix(y_test, predictions)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Confusion Matrix — {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

### Classification Reports

In [ ]:
for name, predictions in models_predictions.items():
    print('=' * 70)
    print(name)
    print('=' * 70)
    print(classification_report(y_test, predictions, target_names=target_names))

## 13. Best-Performing Model

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_accuracy = results_df.iloc[0]['Accuracy']
print(f'Best-performing model: {best_model_name}')
print(f'Test accuracy: {best_accuracy:.2%}')
print('Justification: selected because it achieved the highest test accuracy among the evaluated classifiers.')

## 14. Random Forest Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': iris.feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

display(importance_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance_df, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.show()

## 15. Predict a New Iris Flower

In [ ]:
new_flower = pd.DataFrame(
    [[5.9, 3.0, 5.1, 1.8]],
    columns=iris.feature_names
)
prediction = rf_model.predict(new_flower)[0]
print('Predicted species:', target_names[prediction])

## 16. Conclusion

The Iris dataset was explored and classified using Logistic Regression, KNN, and Random Forest. The models were evaluated using accuracy, confusion matrices, and classification reports containing precision, recall, and F1-score. The model with the highest test accuracy is declared the best-performing model. The visualisations and Random Forest feature importance also show that petal measurements are generally highly useful for distinguishing the three iris species.